Construcción de clasificadores usando inducción de reglas
===

* *60 min*

Las técnicas de clasificación son utilizadas para resolver problemas en los cuales se debe determinar a cual clase pertenece un nuevo conjunto de características; entre los casos prácticos de aplicación se encuentran: el diagnóstico de enfermedades (el paciente tiene o no la enfermedad), la detección de fraudes, los sistemas de reconocimiento (el objeto es o no es una persona), el riesgo crediticio (el solicitante pagará o no pagará la deuda). En este tutorial se presenta el algoritmo 1R, el cual permite construir un clasificador de referencia basado en reglas de asociación, y con el cual se pueden contrastar los resultados de otros algoritmos más complejos. Este tipo de clasificadores pueden ser usados como la línea base para técnicas más complejas.



Al finalizar este tutorial, usted estará en capacidad de:

* Definir el concepto de probabilidad en términos de frecuencia.


* Definir los conceptos de espacio muestral y evento.


* Aplicar los principales axiomas de la probabilidad.


* Explicar que es un sistema de inducción de reglas de asociación para clasificación.


* Explicar el funcionamiento de la metodología 1R.


* Intepretar las principales métricas usadas para evaluar clasificadores.


* Usar el lenguaje R para resolver este tipo de problemas.

## Definición del problema

Teniendo en cuenta un sistema de quejas, se desea determinar si una queja se considera `Procedente`, `reincidente` o `improcedente`, con base en una muestra de observaciones recolectadas previamente sobre el motivo de las quejas, el rango de tiempo después del servicio para presentar la queja y el tipo de registro de la queja. Para ejemplificar la construcción del sistema de clasificación, se tiene un conjunto ficticio de 15 ejemplos para los cuales se tienen las siguientes tres características:

* Motivo de queja ($x_1$):

    - `a`: Mala atención.
    - `b`: No solución del problema.
    - `c`: Mucho tiempo de respuesta.

* Rango de tiempo después del servicio para presentar la queja ($x_2$):

    - `e`: Entre la primera semana.
    - `f`: Entre la 2 y 6 semana.
    - `g`: Después de la 6 semana. 

* Tipo de registro ($x_3$):

    - `h`: Página web.
    - `i`: Registro físico).
    - `j`: Aplicación móvil.

Se desea determinar a que clase pertenece una nueva queja con características $x_1$, $x_2$ y $x_3$. Las clases representan que la queja se considere:
    
- `Procedente` (A).
- `Reincidente` (B).
- `improcedente` (C).


     #   x1   x2   x3    y  
    ------------------------
     1    a    g    h    A
     2    a    e    i    A
     3    a    f    h    A
     4    a    f    j    A
     5    c    g    j    A
     6    b    e    i    B 
     7    b    f    i    B
     8    b    f    i    B
     9    c    g    h    B
    10    c    g    h    B
    11    a    e    i    C
    12    b    g    j    C
    13    c    f    j    C
    14    c    g    h    C
    15    c    e    i    C

## Solución

### Definición de probabilidad como frecuencia

**Espacio muestral:** se define como el conjunto de todos los posibles resultados de un experimento.

**Evento:** Es cualquier colección de posibles resultados de un experimento (subconjunto del espacio muestral).

En su forma más simple, el concepto de probabilidad puede interpretarse como la frecuencia con que ocurre un evento. Por ejemplo, si en la tirada de dos dados se definen como un evento la cantidad de posibles resultados que da la suma de los dos valores resultantes de la tirada, entonces:

<img src="images/bayes/tirada-dados.jpg" width=300>


    Pr(𝑋= 2) = 1 / 36   Pr(𝑋= 6) = 5 / 36   Pr(𝑋=10) = 3 / 36
    Pr(𝑋= 3) = 2 / 36   Pr(𝑋= 7) = 6 / 36   Pr(𝑋=11) = 2 / 36
    Pr(𝑋= 4) = 3 / 36   Pr(𝑋= 8) = 5 / 36   Pr(𝑋=12) = 1 / 36
    Pr(𝑋= 5) = 4 / 36   Pr(𝑋= 9) = 4 / 36

**Ejemplo.--** Los soldados mediavales apostaban con dos dados de la siguiente manera: si el resultado es par {2, 4, 6, 8, 10, 12} ganaba el soldado A; y si el resultado es impar {3, 5, 7, 9, 11} ganaba el soldado B. ¿Quién tiene mayor probabilidad de ganar?

`Rta/` La probabilidad es del 50% para el soldado A y para el soldado B ya que los casos posibles, en términos de conteo, son 5 para ambos casos.

### Propiedades y definiciones básicas


* Todas las probabilidades deben estar entre $0$ y $1$: 


$$0 \le \text{Pr}(x_i) \le 1$$


* Las probabilidades de eventos mutuamente exclusivos (eventos que no pueden ocurrir simultáneamente) y colectivamente exhaustivos (eventos que cubren todo el universo de casos posibles) deben sumar la unidad:

$$\sum_{i=1}^n \text{Pr}(x_i) = 1$$


* Probabilidad condicional $\text{Pr}(A \; | \; B)$: probabilidad de que ocurra un evento $A$ sabiendo que otro evento $B$ ya ocurrio.


* Independencia: Si los eventos $A$ y $B$ son independientes, entonces:

$$\text{Pr}(A \; |  \; B) = \text{Pr}(A)$$

### Reglas de asociación para clasificación

La metodología 1R se basa en la clasificación de los datos de entrada usando una y sólo una de las variables (atributos) del problema. Para el problema planteado, una regla basada en el atributo `x1` podría ser:

    if x1 in {a, d}:  y = A
    if x1 in {b}:     y = B
    if x1 in {c}:     y = C


El algoritmo funciona de la siguiente forma: se toma el primer atributo $x_1$ y se divide en grupos por cada  valor que puede tomar dicho atributo, es decir, por `a`, `b`, `c`, y `d`; para cada atributo se determina a que clase es más probable que pertenezca las observaciones y se asigna dicha clase a dicho atributo. Es decir, para cada atributo se cuentan cuántos ejemplos hay de cada categoría y se asigna la clase por mayoría, esto es, si hay cuatro ejemplos para la categoría `a` de $x_1$ y tres de ellos pertencen a la clase `A` y el restante a `C` se dice que `if x1 == a: y = A`; esto equivale a decir que si `x1 == a` es más probable que el ejemplo pertenezca a la clase sea `A`. Así, el clasificador basado en este atributo podría ser escrito como un sistema de reglas:

         #   x1   x2   x3    y  
    --------------------------

    if x1 == a:  y = A
    
         1    a    g    h    A
         2    a    e    i    A
         3    a    f    h    A
         4    a    f    j    A
        11    a    e    i    C
        
    if x1 == b:  y = B
    
         6    b    e    i    B 
         7    b    f    i    B
        12    b    g    j    C
         8    b    f    i    B

    if x1 == c:  y = C
    
         5    d    g    j    A
         9    c    g    h    B
        10    c    g    h    B        
        13    c    f    j    C
        14    c    g    h    C
        15    c    e    i    C         

       
Al agrupar por $x_1$, este conjunto de reglas se reescribe como:

    if x1 in {a}:
        y = A
    elif x1 in {b}:
        y = B
    else:
        y = C

Luego se toma el segundo atributo $x_2$ y se procede de igual forma para construir otro clasificador. El proceso se repite hasta obtener un clasificador por cada atributo. Se escoge el clasificador con mayor precisión.

---
**Ejemplo.--** Para el caso anterior, escriba las reglas para el clasificador para el atributo $x_2$.
    
    
         #   x1   x2   x3    y  
    --------------------------   
    
    if x2 == e:  y = C
    
         2    a    e    i    A
         6    b    e    i    B 
        11    a    e    i    C
        15    c    e    i    C

Como se muestra a continuación, usualmente el conteo de clases resulta igual para todas o algunas de ellas, es decir, hay dos casos para la clase A y dos casos para la clase B. Se escoge indiferentemenete entre A o B ya que ambas tienen la misma probabilidad, en términos de conteo, de ser escogidas.

         #   x1   x2   x3    y  
    --------------------------    
       
    if x2 == f:  y = A
    
         3    a    f    h    A
         4    d    f    j    A
         7    b    f    i    B
         8    b    f    i    B
        13    c    f    j    C

    if x2 == g:  y = B
    
         1    a    g    h    A
         5    c    g    j    A
         9    c    g    h    B
        10    c    g    h    B
        12    b    g    j    C
        14    c    g    h    C

       
Al agrupar por $x_2$, este conjunto de reglas se reescribe como:

    if x2 in {e}:
        y = C
    elif x2 in {f}:
        y = A
    else:
        y = B
        
---

### Métricas de desempeño de clasificadores

Para evaluar el desempeño en problemas de clasificación dicotómicos (dos clases mutuamente excluyentes) se usa la siguiente tabla:


             | Pronostico
             |  P     N
    ---------|------------  
          P  |  TP    FN          
    Real     |
          N  |  FP    TN
    
    TP - Verdadero positivo (correcto)
    TN - Verdadero negativo (correcto)
    FN - Falso negativo (mal clasificado)
    FP - Falso positivo (mal clasificado)
    
La medición de la precisión del modelo permite estimar el desempeño del modelo ante nuevos datos.

* Tasa de éxito o accuracy (porcentaje de patrones clasificados correctamente):


$$\text{success rate} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}$$

* Tasa de error (porcentaje total de patrones clasificados incorrectamente):

$$\text{error rate} = \frac{\text{FP} + \text{FN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}} = 1 - \text{accuracy}$$

* Precisión o valor predictivo positivo: Proporción de casos positivos que fueron verdaderamente positivos.


$$\text{precision} = \frac{\text{TP}}{\text{TP}  + \text{FP}}$$

* Valor predictivo negativo: Proporción de casos negativos que fueron verdaderamente negativos.


$$\text{negative predictive value} = \frac{\text{TN}}{\text{TN}  + \text{FN}}$$

* Sensibilidad, tasa verdadera positiva, recall: mide la proporción de ejemplos positivos que fueron correctamente clasificados.


$$\text{sensitibity} = \frac{\text{TP}}{\text{TP}  + \text{FN}}$$

* Especifidad o tasa verdadera negativa: mide la proporción de ejemplos negativos correctamente clasificados.


$$\text{specifity} = \frac{\text{TN}}{\text{TN}  + \text{FP}}$$

---
**Ejemplo.--** Para el clasificador basado en $x_1$, calcule las métricas de error descritas.

Note que las metricas descritas anteriormente están basadas en la matriz de confusión para clases dicotómicas. En este caso, las clases están compuestas tres instancias (A,B y C) por lo que se hace necesario definir una de ellas como verdadera, con el objetivo de que las métricas guarden coherencia e interpretabilidad con respecto al problema.


     #   x1   x2   x3    y    y pronosticado  
    ----------------------------------------
     1    a    g    h    A        A
     2    a    e    i    A        A
     3    a    f    h    A        A
     4    a    f    j    A        A
     5    c    g    j    A        C
     6    b    e    i    B        B
     7    b    f    i    B        B
     8    b    f    i    B        B
     9    c    g    h    B        C
    10    c    g    h    B        C
    11    a    e    i    C        A
    12    b    g    j    C        B
    13    c    f    j    C        C
    14    c    g    h    C        C
    15    c    e    i    C        C


             |    Pronostico
             |  A      B     C
    ---------|----------------- 
          A  |  4     0      1   
             |
     Real B  |  0     3      2
             |
          C  |  1     1      3  
          

Para el calculo, se define A como verdadero.

* Tasa de Éxito =  (4+3+3)/15 = 66,66%

* Tasa de Error =  1 - Tasa de Éxito = 33.33%

* Precisión = 4/(4+0+1+1) = 66.66%

* Valor predictivo negativo = (3+3)/(3+3+1+2+0) =  66.66%

* Sensibilidad = 4/(3+3+1+2+0) = 44.44%

* Especificidad = (3+3)/(3+3+0+1+1) = 75%

---

### Solución en el lenguaje R

El sistema de asociación de reglas está implementado en la librearía `OneR`.

In [6]:
##
## Se preparan los datos
##
x1 <- c('a', 'a', 'a', 'a', 'c', 'b', 'b', 'b', 'c', 'c', 'a', 'b', 'c', 'c', 'c')
x2 <- c('g', 'e', 'f', 'f', 'g', 'e', 'f', 'f', 'g', 'g', 'e', 'g', 'f', 'g', 'e')
x3 <- c('h', 'i', 'h', 'j', 'j', 'i', 'i', 'i', 'h', 'h', 'i', 'j', 'j', 'h', 'i')
y  <- c('A', 'A', 'A', 'A', 'A', 'B', 'B', 'B', 'B', 'B', 'C', 'C', 'C', 'C', 'C')

##
## Se crea un dataframe con los datos como factores
##
data <- data.frame(x1=factor(x1), 
                   x2=factor(x2), 
                   x3=factor(x3), 
                   y=factor(y))

In [7]:
##
## Instalación del paquete
##

# install.packages("OneR")

In [8]:
##
## Carga la librería.
##
library(OneR)

##
## Crea el clasificador. La librería reporta la precisión 
## del clasificador si se usa cada uno de los atributos 
## (variables x) y el sistema de reglas obtenido. 
## La notación y ~ . indica que la variable y del dataframe es función de las restantes
##
clf <- OneR(y ~ ., data = data, verbose = TRUE)

##
## la salida del modelo indica la tasa de éxito para los clasificadores de todas las clases
## Igualmente se imprimen las reglas de decisión para el mejor clasificador
##
clf  


    Attribute Accuracy
1 * x1        66.67%  
2   x3        46.67%  
3   x2        40%     
---
Chosen attribute due to accuracy
and ties method (if applicable): '*'




Call:
OneR.formula(formula = y ~ ., data = data, verbose = TRUE)

Rules:
If x1 = a then y = A
If x1 = b then y = B
If x1 = c then y = C

Accuracy:
10 of 15 instances classified correctly (66.67%)


In [9]:
##
## La función summary reporta información detallada
## de los resultados del modelo, junto con la matriz de confusión
##
summary(clf)


Call:
OneR.formula(formula = y ~ ., data = data, verbose = TRUE)

Rules:
If x1 = a then y = A
If x1 = b then y = B
If x1 = c then y = C

Accuracy:
10 of 15 instances classified correctly (66.67%)

Contingency table:
     x1
y       a   b   c Sum
  A   * 4   0   1   5
  B     0 * 3   2   5
  C     1   1 * 3   5
  Sum   5   4   6  15
---
Maximum in each column: '*'

Pearson's Chi-squared test:
X-squared = 9.7, df = 4, p-value = 0.0458



---